In [ ]:


import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
Cell 2 — Upload data (run once)


import os
os.makedirs("data/raw/en", exist_ok=True)
os.makedirs("data/raw/es", exist_ok=True)

from google.colab import files

print("Upload EN: train.conll, dev.conll, test.conll")
uploaded = files.upload()
for fname, content in uploaded.items():
    with open(f"data/raw/en/{fname}", "wb") as f: f.write(content)

print("Upload ES: train.conll, dev.conll, test.conll")
uploaded = files.upload()
for fname, content in uploaded.items():
    with open(f"data/raw/es/{fname}", "wb") as f: f.write(content)
Cell 3A — Option A: xlm-roberta-base joint EN+ES


!python src/train.py \
    --lang joint \
    --model-name xlm-roberta-base \
    --batch-size 16 \
    --weight-cap 5.0 \
    --focal-gamma 0.0 \
    --epochs 8 \
    --lr 2e-5 \
    --label-smoothing 0.1
Cell 3B — Option B: Biomedical EN model


!python src/train.py \
    --lang en \
    --model-name allenai/biomed_roberta_base \
    --batch-size 16 \
    --weight-cap 5.0 \
    --focal-gamma 0.0 \
    --epochs 8 \
    --lr 2e-5 \
    --label-smoothing 0.1 \
    --reprocess
Cell 3C — Option C: Spanish clinical model


!python src/train.py \
    --lang es \
    --model-name PlanTL-GOB-ES/roberta-base-biomedical-clinical-es \
    --batch-size 16 \
    --weight-cap 5.0 \
    --focal-gamma 0.0 \
    --epochs 8 \
    --lr 2e-5 \
    --label-smoothing 0.1 \
    --reprocess
Cell 4 — Evaluate all models


import sys, shutil, os
sys.path.insert(0, "src")
from evaluate import evaluate

# After Option A (joint model — test both languages)
shutil.copytree("outputs/checkpoints/best_model", "outputs/checkpoints/optionA", dirs_exist_ok=True)
evaluate("outputs/checkpoints/optionA", lang="en")
shutil.copy("outputs/results/en_test_results.json", "outputs/results/optionA_en.json")
evaluate("outputs/checkpoints/optionA", lang="es")
shutil.copy("outputs/results/es_test_results.json", "outputs/results/optionA_es.json")

# After Option B (EN biomedical model)
shutil.copytree("outputs/checkpoints/best_model", "outputs/checkpoints/optionB", dirs_exist_ok=True)
evaluate("outputs/checkpoints/optionB", lang="en")
shutil.copy("outputs/results/en_test_results.json", "outputs/results/optionB_en.json")

# After Option C (ES clinical model)
shutil.copytree("outputs/checkpoints/best_model", "outputs/checkpoints/optionC", dirs_exist_ok=True)
evaluate("outputs/checkpoints/optionC", lang="es")
shutil.copy("outputs/results/es_test_results.json", "outputs/results/optionC_es.json")
Cell 5 — Comparison table


import json

def load(p):
    try:
        return json.load(open(p))
    except: return {}

def row(name, lang, path):
    d = load(path)
    return (name, lang,
            round(d.get("DIS",{}).get("f1-score",0),4),
            round(d.get("SYM",{}).get("f1-score",0),4),
            round(d.get("PRO",{}).get("f1-score",0),4),
            round(d.get("micro avg",{}).get("f1-score",0),4))

rows = [
    ("Run 02 baseline","EN", 0.3747,0.2981,0.3167,0.3304),
    ("Run 02 baseline","ES", 0.3692,0.3172,0.3791,0.3551),
    row("Option A (XLM-R joint)","EN","outputs/results/optionA_en.json"),
    row("Option A (XLM-R joint)","ES","outputs/results/optionA_es.json"),
    row("Option B (biomed EN)",  "EN","outputs/results/optionB_en.json"),
    row("Option C (clinical ES)","ES","outputs/results/optionC_es.json"),
]

print(f"{'Model':<30} {'Lang':<5} {'DIS':>7} {'SYM':>7} {'PRO':>7} {'Overall':>9}")
print("-"*65)
for r in rows:
    print(f"{r[0]:<30} {r[1]:<5} {r[2]:>7.4f} {r[3]:>7.4f} {r[4]:>7.4f} {r[5]:>9.4f}")
Cell 6 — Download results


from google.colab import files
import shutil
shutil.make_archive("results", "zip", "outputs/results")
files.download("results.zip")
Note: Run 3A → 3B → 3C one at a time — each takes ~45–60 min on T4. Run Cell 4 evaluation after each training finishes (not all at once), otherwise best_model gets overwritten.

In [ ]:
!git fetch origin
!git checkout option-6
!git pull origin option-6
!pip install -q -r requirements.txt


remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (5/5), done.
Unpacking objects: 100% (9/9), 9.59 KiB | 3.20 MiB/s, done.
remote: Total 9 (delta 4), reused 9 (delta 4), pack-reused 0 (from 0)
From https://github.com/azharthegeek/MultiClinNER-XLMR
   7020a1a..db12770  option-6   -> origin/option-6
Already on 'option-6'
Your branch is behind 'origin/option-6' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/azharthegeek/MultiClinNER-XLMR
 * branch            option-6   -> FETCH_HEAD
Updating 7020a1a..db12770
Fast-forward
 NEXT_STEPS.md   | 309 ++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 run_all.sh      | 136 +++++++++++++++++++++++++
 src/model.py    |   6 +-
 src/negation.py | 108 ++++++++++++++++++++
 src/train.py    | 106 ++++++++++++++-----
 src/utils.py    |  70 +++++++++----
 6 files changed, 688 insertions(+), 47 deletions(-)
 create mod

In [ ]:
%cd MultiClinNER-XLMR
!pip install -q -r requirements.txt

import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


/content/MultiClinNER-XLMR
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.3 MB/s eta 0:00:00
CUDA: True
GPU: Tesla T4


In [ ]:
import os
os.makedirs("data/raw/en", exist_ok=True)
os.makedirs("data/raw/es", exist_ok=True)

from google.colab import files

print("Upload EN: train.conll, dev.conll, test.conll")
uploaded = files.upload()
for fname, content in uploaded.items():
    with open(f"data/raw/en/{fname}", "wb") as f: f.write(content)

print("Upload ES: train.conll, dev.conll, test.conll")
uploaded = files.upload()
for fname, content in uploaded.items():
    with open(f"data/raw/es/{fname}", "wb") as f: f.write(content)


### Mount Google Drive and copy data

I will mount Google Drive to access your data and copy the `NLP-Training-data/data/raw` folder to the `MultiClinNER-XLMR/data/raw` directory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil

source_path = '/content/drive/MyDrive/NLP-Training-data/data/raw'
destination_path = 'data/raw'

# Ensure the destination directory exists (it should from previous steps)
# os.makedirs(destination_path, exist_ok=True)

# Copy contents from source to destination
shutil.copytree(source_path, destination_path, dirs_exist_ok=True)

print(f'Contents from {source_path} copied to {destination_path}')

Contents from /content/drive/MyDrive/NLP-Training-data/data/raw copied to data/raw


In [ ]:
!python src/train.py \
    --lang joint \
    --model-name xlm-roberta-base \
    --batch-size 16 \
    --weight-cap 5.0 \
    --focal-gamma 0.0 \
    --epochs 8 \
    --lr 2e-5 \
    --label-smoothing 0.1


model.safetensors: 100% 1.12G/1.12G [00:06<00:00, 171MB/s]
Loading weights: 100% 197/197 [00:00<00:00, 12776.24it/s]
[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Class weights: {'O': '0.163', 'B-DIS': '0.815', 'I-DIS': '0.815', 'B

In [ ]:
!python src/train.py \
    --lang en \
    --model-name allenai/biomed_roberta_base \
    --batch-size 16 \
    --weight-cap 5.0 \
    --focal-gamma 0.0 \
    --epochs 8 \
    --lr 2e-5 \
    --label-smoothing 0.1 \
    --reprocess


Tokenizing dataset with allenai/biomed_roberta_base ...
Map:   0% 0/10000 [00:00<?, ? examples/s]
config.json: 100% 430/430 [00:00<00:00, 2.26MB/s]

tokenizer_config.json: 100% 185/185 [00:00<00:00, 1.24MB/s]

vocab.json: 100% 899k/899k [00:00<00:00, 82.1MB/s]

merges.txt: 100% 456k/456k [00:00<00:00, 93.7MB/s]

added_tokens.json: 100% 2.00/2.00 [00:00<00:00, 13.4kB/s]

special_tokens_map.json: 100% 150/150 [00:00<00:00, 1.06MB/s]
Map: 100% 10000/10000 [00:14<00:00, 698.17 examples/s]
Map: 100% 2315/2315 [00:02<00:00, 962.79 examples/s]
Map: 100% 2250/2250 [00:02<00:00, 943.87 examples/s]
Saving the dataset (1/1 shards): 100% 10000/10000 [00:00<00:00, 513548.42 examples/s]
Saving the dataset (1/1 shards): 100% 2315/2315 [00:00<00:00, 364948.27 examples/s]
Saving the dataset (1/1 shards): 100% 2250/2250 [00:00<00:00, 344875.90 examples/s]
Saved processed dataset to /content/MultiClinNER-XLMR/data/processed/en_allenai_biomed_roberta_base
pytorch_model.bin: 100% 656M/656M [00:05<00:00, 11

In [ ]:
!python src/train.py \
    --lang es \
    --model-name PlanTL-GOB-ES/roberta-base-biomedical-clinical-es \
    --batch-size 16 \
    --weight-cap 5.0 \
    --focal-gamma 0.0 \
    --epochs 8 \
    --lr 2e-5 \
    --label-smoothing 0.1 \
    --reprocess


Tokenizing dataset with PlanTL-GOB-ES/roberta-base-biomedical-clinical-es ...
Map:   0% 0/10163 [00:00<?, ? examples/s]
config.json: 100% 613/613 [00:00<00:00, 3.00MB/s]

tokenizer_config.json: 100% 1.27k/1.27k [00:00<00:00, 5.23MB/s]

vocab.json: 100% 1.22M/1.22M [00:00<00:00, 44.4MB/s]

merges.txt: 100% 540k/540k [00:00<00:00, 92.2MB/s]

special_tokens_map.json: 100% 772/772 [00:00<00:00, 4.64MB/s]
Map: 100% 10163/10163 [00:16<00:00, 625.06 examples/s]
Map: 100% 2151/2151 [00:02<00:00, 778.92 examples/s]
Map: 100% 2251/2251 [00:02<00:00, 791.70 examples/s]
Saving the dataset (1/1 shards): 100% 10163/10163 [00:00<00:00, 396793.31 examples/s]
Saving the dataset (1/1 shards): 100% 2151/2151 [00:00<00:00, 226642.25 examples/s]
Saving the dataset (1/1 shards): 100% 2251/2251 [00:00<00:00, 270969.16 examples/s]
Saved processed dataset to /content/MultiClinNER-XLMR/data/processed/es_PlanTL-GOB-ES_roberta-base-biomedical-clinical-es
pytorch_model.bin: 100% 504M/504M [00:04<00:00, 111MB/s]
Lo

In [ ]:
import sys, shutil, os
sys.path.insert(0, "src")
from evaluate import evaluate

# After Option A (joint model — test both languages)
shutil.copytree("outputs/checkpoints/best_model", "outputs/checkpoints/optionA", dirs_exist_ok=True)
evaluate("outputs/checkpoints/optionA", lang="en")
shutil.copy("outputs/results/en_test_results.json", "outputs/results/optionA_en.json")
evaluate("outputs/checkpoints/optionA", lang="es")
shutil.copy("outputs/results/es_test_results.json", "outputs/results/optionA_es.json")

# After Option B (EN biomedical model)
shutil.copytree("outputs/checkpoints/best_model", "outputs/checkpoints/optionB", dirs_exist_ok=True)
evaluate("outputs/checkpoints/optionB", lang="en")
shutil.copy("outputs/results/en_test_results.json", "outputs/results/optionB_en.json")

# After Option C (ES clinical model)
shutil.copytree("outputs/checkpoints/best_model", "outputs/checkpoints/optionC", dirs_exist_ok=True)
evaluate("outputs/checkpoints/optionC", lang="es")
shutil.copy("outputs/results/es_test_results.json", "outputs/results/optionC_es.json")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
cd MultiClinNER-XLMR/

/content/MultiClinNER-XLMR


In [ ]:
!git pull origin option-6


remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.07 KiB | 1.03 MiB/s, done.
From https://github.com/azharthegeek/MultiClinNER-XLMR
 * branch            option-6   -> FETCH_HEAD
   8192798..5f1d6af  option-6   -> origin/option-6
Updating 8192798..5f1d6af
Fast-forward
 src/evaluate.py | 94 ++++++++++++++++++++++++++++++++++++++++++++++-----------
 1 file changed, 76 insertions(+), 18 deletions(-)


In [ ]:
!python src/evaluate.py --model outputs/checkpoints/optionA --lang en --device cpu


Tokenizer : /content/MultiClinNER-XLMR/outputs/checkpoints/optionA  (vocab_size=52000)
Tokenizing /content/MultiClinNER-XLMR/data/raw/en/test.conll ...
Loading model from /content/MultiClinNER-XLMR/outputs/checkpoints/optionA ...
Loading weights: 100% 199/199 [00:00<00:00, 6563.80it/s]
Model     : vocab_size=52000  num_labels=7
Device    : cpu
Running inference ...
Traceback (most recent call last):
  File "/content/MultiClinNER-XLMR/src/evaluate.py", line 307, in <module>
    evaluate(args.model, args.lang, args.split, model_name=args.model_name, device=_device)
  File "/content/MultiClinNER-XLMR/src/evaluate.py", line 184, in evaluate
    all_preds = _run_inference(model, tokenizer, all_input_ids, all_attention_masks, device=device)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/MultiClinNER-XLMR/src/evaluate.py", line 109, in _run_inference
    logits = model(input_ids=padded_ids, attention_mask=padded_masks).logi

In [ ]:
# Stop the evaluate cell, then run this
import json
cfg = json.load(open("outputs/checkpoints/optionA/config.json"))
print("model_type  :", cfg.get("model_type"))
print("vocab_size  :", cfg.get("vocab_size"))
print("_name_or_path:", cfg.get("_name_or_path"))


model_type  : roberta
vocab_size  : 52000
_name_or_path: None


In [ ]:
# Cell 1 — Clear the stale checkpoint
import shutil, os
for d in ["outputs/checkpoints/best_model", "outputs/checkpoints/optionA"]:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Removed {d}")



Removed outputs/checkpoints/best_model
Removed outputs/checkpoints/optionA


In [ ]:
# Cell 2 — Train Option A (xlm-roberta-base joint)
!python src/train.py \
    --lang joint \
    --model-name xlm-roberta-base \
    --weight-cap 5.0 --focal-gamma 0.0 \
    --epochs 8 --lr 2e-5 --label-smoothing 0.1 \
    --reprocess




Tokenizing dataset with xlm-roberta-base ...
Map: 100% 10000/10000 [00:13<00:00, 719.49 examples/s]
Map: 100% 2315/2315 [00:02<00:00, 879.30 examples/s]
Map: 100% 2250/2250 [00:03<00:00, 647.33 examples/s]
Map: 100% 10163/10163 [00:17<00:00, 593.97 examples/s]
Map: 100% 2151/2151 [00:03<00:00, 644.66 examples/s]
Map: 100% 2251/2251 [00:03<00:00, 672.72 examples/s]
Saving the dataset (1/1 shards): 100% 20163/20163 [00:00<00:00, 296936.01 examples/s]
Saving the dataset (1/1 shards): 100% 2315/2315 [00:00<00:00, 283539.60 examples/s]
Saving the dataset (1/1 shards): 100% 2250/2250 [00:00<00:00, 275626.74 examples/s]
Saved processed dataset to /content/MultiClinNER-XLMR/data/processed/joint
Loading weights: 100% 197/197 [00:00<00:00, 1079.55it/s]
[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UN

In [ ]:
# Cell 3 — Copy to optionA slot
import shutil
shutil.copytree("outputs/checkpoints/best_model", "outputs/checkpoints/optionA", dirs_exist_ok=True)
print("Done")

Done


In [ ]:
!python src/evaluate.py --model outputs/checkpoints/optionA --lang en
!python src/evaluate.py --model outputs/checkpoints/optionA --lang es


Tokenizer : /content/MultiClinNER-XLMR/outputs/checkpoints/optionA  (vocab_size=250002)
Tokenizing /content/MultiClinNER-XLMR/data/raw/en/test.conll ...
Loading model from /content/MultiClinNER-XLMR/outputs/checkpoints/optionA ...
Loading weights: 100% 199/199 [00:00<00:00, 5350.77it/s]
Model     : vocab_size=250002  num_labels=7
Device    : cuda
Running inference ...

=== EN Test Results ===
              precision    recall  f1-score   support

         DIS       0.25      0.67      0.36      3087
         PRO       0.20      0.71      0.31      2537
         SYM       0.18      0.61      0.28      2754

   micro avg       0.21      0.66      0.32      8378
   macro avg       0.21      0.66      0.32      8378
weighted avg       0.21      0.66      0.32      8378

Overall F1 : 0.3170
Traceback (most recent call last):
  File "/content/MultiClinNER-XLMR/src/evaluate.py", line 307, in <module>
    evaluate(args.model, args.lang, args.split, model_name=args.model_name, device=_device)
 

In [ ]:
import shutil
shutil.copytree("outputs/checkpoints/best_model", "outputs/checkpoints/optionA", dirs_exist_ok=True)

!python src/evaluate.py --model outputs/checkpoints/optionA --lang en
!cp outputs/results/en_test_results.json outputs/results/optionA_en.json

!python src/evaluate.py --model outputs/checkpoints/optionA --lang es
!cp outputs/results/es_test_results.json outputs/results/optionA_es.json


Tokenizer : /content/MultiClinNER-XLMR/outputs/checkpoints/optionA  (vocab_size=250002)
Tokenizing /content/MultiClinNER-XLMR/data/raw/en/test.conll ...
Loading model from /content/MultiClinNER-XLMR/outputs/checkpoints/optionA ...
Loading weights: 100% 199/199 [00:00<00:00, 4554.13it/s]
Model     : vocab_size=250002  num_labels=7
Device    : cuda
Running inference ...

=== EN Test Results ===
              precision    recall  f1-score   support

         DIS       0.25      0.67      0.36      3087
         PRO       0.20      0.71      0.31      2537
         SYM       0.18      0.61      0.28      2754

   micro avg       0.21      0.66      0.32      8378
   macro avg       0.21      0.66      0.32      8378
weighted avg       0.21      0.66      0.32      8378

Overall F1 : 0.3170
Traceback (most recent call last):
  File "/content/MultiClinNER-XLMR/src/evaluate.py", line 307, in <module>
    evaluate(args.model, args.lang, args.split, model_name=args.model_name, device=_device)
 

In [ ]:
shutil.copytree("outputs/checkpoints/best_model", "outputs/checkpoints/optionB", dirs_exist_ok=True)
!python src/evaluate.py --model outputs/checkpoints/optionB --lang en
!cp outputs/results/en_test_results.json outputs/results/optionB_en.json


Tokenizer : /content/MultiClinNER-XLMR/outputs/checkpoints/optionB  (vocab_size=250002)
Tokenizing /content/MultiClinNER-XLMR/data/raw/en/test.conll ...
Loading model from /content/MultiClinNER-XLMR/outputs/checkpoints/optionB ...
Loading weights: 100% 199/199 [00:00<00:00, 5243.90it/s]
Model     : vocab_size=250002  num_labels=7
Device    : cuda
Running inference ...

=== EN Test Results ===
              precision    recall  f1-score   support

         DIS       0.25      0.67      0.36      3087
         PRO       0.20      0.71      0.31      2537
         SYM       0.18      0.61      0.28      2754

   micro avg       0.21      0.66      0.32      8378
   macro avg       0.21      0.66      0.32      8378
weighted avg       0.21      0.66      0.32      8378

Overall F1 : 0.3170
Traceback (most recent call last):
  File "/content/MultiClinNER-XLMR/src/evaluate.py", line 307, in <module>
    evaluate(args.model, args.lang, args.split, model_name=args.model_name, device=_device)
 

In [ ]:
shutil.copytree("outputs/checkpoints/best_model", "outputs/checkpoints/optionC", dirs_exist_ok=True)
!python src/evaluate.py --model outputs/checkpoints/optionC --lang es
!cp outputs/results/es_test_results.json outputs/results/optionC_es.json


Tokenizer : /content/MultiClinNER-XLMR/outputs/checkpoints/optionC  (vocab_size=250002)
Tokenizing /content/MultiClinNER-XLMR/data/raw/es/test.conll ...
Loading model from /content/MultiClinNER-XLMR/outputs/checkpoints/optionC ...
Loading weights: 100% 199/199 [00:00<00:00, 19915.69it/s]
Model     : vocab_size=250002  num_labels=7
Device    : cuda
Running inference ...

=== ES Test Results ===
              precision    recall  f1-score   support

         DIS       0.24      0.71      0.35      2692
         PRO       0.25      0.74      0.37      2959
         SYM       0.19      0.66      0.29      2584

   micro avg       0.22      0.71      0.34      8235
   macro avg       0.22      0.70      0.34      8235
weighted avg       0.22      0.71      0.34      8235

Overall F1 : 0.3379
Traceback (most recent call last):
  File "/content/MultiClinNER-XLMR/src/evaluate.py", line 307, in <module>
    evaluate(args.model, args.lang, args.split, model_name=args.model_name, device=_device)


In [ ]:
import json

def load(p):
    try:
        return json.load(open(p))
    except: return {}

def row(name, lang, path):
    d = load(path)
    return (name, lang,
            round(d.get("DIS",{}).get("f1-score",0),4),
            round(d.get("SYM",{}).get("f1-score",0),4),
            round(d.get("PRO",{}).get("f1-score",0),4),
            round(d.get("micro avg",{}).get("f1-score",0),4))

rows = [
    ("Run 02 baseline","EN", 0.3747,0.2981,0.3167,0.3304),
    ("Run 02 baseline","ES", 0.3692,0.3172,0.3791,0.3551),
    row("Option A (XLM-R joint)","EN","outputs/results/optionA_en.json"),
    row("Option A (XLM-R joint)","ES","outputs/results/optionA_es.json"),
    row("Option B (biomed EN)",  "EN","outputs/results/optionB_en.json"),
    row("Option C (clinical ES)","ES","outputs/results/optionC_es.json"),
]

print(f"{'Model':<30} {'Lang':<5} {'DIS':>7} {'SYM':>7} {'PRO':>7} {'Overall':>9}")
print("-"*65)
for r in rows:
    print(f"{r[0]:<30} {r[1]:<5} {r[2]:>7.4f} {r[3]:>7.4f} {r[4]:>7.4f} {r[5]:>9.4f}")


Model                          Lang      DIS     SYM     PRO   Overall
-----------------------------------------------------------------
Run 02 baseline                EN     0.3747  0.2981  0.3167    0.3304
Run 02 baseline                ES     0.3692  0.3172  0.3791    0.3551
Option A (XLM-R joint)         EN     0.0000  0.0000  0.0000    0.0000
Option A (XLM-R joint)         ES     0.0000  0.0000  0.0000    0.0000
Option B (biomed EN)           EN     0.0000  0.0000  0.0000    0.0000
Option C (clinical ES)         ES     0.0000  0.0000  0.0000    0.0000


In [ ]:
from google.colab import files
import shutil
shutil.make_archive("results", "zip", "outputs/results")
files.download("results.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>